In [236]:
!python -m spacy download en_core_web_lg

     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     --------------------------------------- 3.1/400.7 MB 26.7 MB/s eta 0:00:15
      ------------------------------------- 10.5/400.7 MB 39.7 MB/s eta 0:00:10
     - ------------------------------------ 17.8/400.7 MB 33.6 MB/s eta 0:00:12
     -- ----------------------------------- 22.8/400.7 MB 30.4 MB/s eta 0:00:13
     -- ----------------------------------- 31.5/400.7 MB 36.6 MB/s eta 0:00:11
     --- ---------------------------------- 41.2/400.7 MB 34.8 MB/s eta 0:00:11
     ---- --------------------------------- 52.4/400.7 MB 37.7 MB/s eta 0:00:10
     ----- -------------------------------- 62.9/400.7 MB 39.7 MB/s eta 0:00:09
     ------ ------------------------------- 73.4/400.7 MB 41.1 MB/s eta 0:00:08
     ------- ------------------------------ 83.9/400.7 MB 42.2 MB/s eta 0:00:08
     -------- ----------------------------- 94.4/400.7 MB 43.0 MB/s eta 0:00:08
     --------- ---------------------------- 94.

In [237]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [238]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [239]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [240]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [241]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [242]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [248]:
import spacy

nlp = spacy.load("en_core_web_lg")

In [334]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti (osebe, kraji, jeziki, narodi)
    odstrani_pos = ['PROPN', 'PRON', 'VERB', 'ADV']
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    # ustvarimo množico besed, ki so del prepoznanih entitet (imen)
    

    for token in doc:
        # če je beseda ime
        if token.pos_ in odstrani_pos:
            continue
        if token.ent_type_ in odstrani_entitete:
            continue
            
        # spaCy uporablja oznake: NOUN ADJ
        if token.pos_ in ['NOUN']:
            beseda = token.lemma_.lower()
            # samo črke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [335]:
import random
from sklearn.feature_extraction import text

In [387]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    # založniški šum 
    'book', 'novel', 'story', 'author', 'literature', 'edition', 'seller', 
    'read', 'reader', 'page', 'chapter', 'write', 'writer', 'publish', 
    'publication', 'review', 'times', 'york', 'print', 'copy', 'best', 
    'original', 'classic', 'introduction', 'note', 'debut', 'thriller',
    'unique', 'fascinating', 'scandal', 'major', 'character', 'cover', 'magazine',
    'self', 'series', 'volume', 'masterpiece', 'translation', 'film', 'tale', 'course',
    
    # splošni opisi 
    'way', 'thing', 'important', 'practical', 'young', 'boy', 'girl', 
    'human', 'people', 'great', 'good', 'bad', 'true', 'new', 'old',
    'life', 'world', 'everything', 'day', 'time', 'year', 'make',
    'take', 'come', 'think', 'feel', 'know', 'look', 'want', 'large', 'small',
    'man', 'woman', 'literary', 'secret', 'isbn', 'mother', 'sister', 'father',
    'little', 'room', 'place', 'end', 'first', 'second',
    
    #iz izpisa
    'professional', 'guide', 'experience', 'natural', 'vivid', 'narrative',
    'compelling', 'extraordinary', 'powerful', 'voice', 'mind'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [390]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\najbolj_popularne\naj_ang_opisi\*.txt')[:250]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words=all_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.75)

In [391]:
X = vectorizer.fit_transform(filepaths) 


In [392]:
words = vectorizer.get_feature_names_out()
freq = np.asarray(X.sum(axis=0)).flatten()

top_words = [words[i] for i in freq.argsort()[-50:]]
print(top_words)

['fiction', 'quest', 'vampire', 'game', 'letter', 'police', 'nature', 'force', 'student', 'brother', 'word', 'summer', 'childhood', 'question', 'wife', 'land', 'hand', 'history', 'horror', 'son', 'earth', 'blood', 'city', 'event', 'job', 'killer', 'crime', 'daughter', 'journey', 'school', 'power', 'century', 'generation', 'truth', 'age', 'home', 'night', 'murder', 'town', 'house', 'heart', 'work', 'war', 'death', 'child', 'adventure', 'past', 'friend', 'family', 'love']


In [393]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3364 stored elements and shape (250, 572)>
  Coords	Values
  (0, 190)	1
  (0, 124)	3
  (0, 243)	2
  (0, 196)	3
  (0, 289)	1
  (0, 12)	1
  (0, 180)	1
  (0, 497)	2
  (0, 336)	1
  (0, 78)	1
  (0, 241)	1
  (0, 296)	1
  (1, 296)	2
  (1, 571)	1
  (1, 62)	1
  (1, 66)	1
  (1, 224)	1
  (1, 244)	1
  (1, 325)	1
  (2, 336)	1
  (2, 458)	1
  (2, 557)	1
  (2, 392)	1
  (2, 568)	1
  (2, 249)	1
  :	:
  (248, 116)	1
  (248, 173)	2
  (248, 162)	1
  (248, 8)	1
  (248, 202)	2
  (248, 7)	1
  (248, 231)	1
  (248, 384)	1
  (248, 489)	1
  (249, 9)	1
  (249, 283)	2
  (249, 464)	1
  (249, 220)	1
  (249, 22)	1
  (249, 345)	1
  (249, 563)	1
  (249, 565)	1
  (249, 529)	1
  (249, 204)	1
  (249, 564)	1
  (249, 31)	1
  (249, 290)	1
  (249, 52)	1
  (249, 46)	1
  (249, 341)	1
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 0]
 ...
 [2 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
   academy  accident  account  achievement  act  action  actress  ada

In [394]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [402]:
# test 

W, H, errors = nmf(X, 4)
print(errors)
#print(W)
#print(H)

[np.float64(69.09817386670636), np.float64(68.27335120628337), np.float64(67.59314884746307), np.float64(67.1481507522208), np.float64(66.85192301804952), np.float64(66.62904239483706), np.float64(66.4591883542499), np.float64(66.33639592317408), np.float64(66.24799986361694), np.float64(66.18230677898319), np.float64(66.13149986394905), np.float64(66.09137671301086), np.float64(66.05974864026399), np.float64(66.03442585772305), np.float64(66.01358949532532), np.float64(65.99631462617184), np.float64(65.98236366123814), np.float64(65.97130141652784), np.float64(65.96226789049473), np.float64(65.95468471717648), np.float64(65.94809314398964), np.float64(65.94208888773632), np.float64(65.93646543384243), np.float64(65.93123121301511), np.float64(65.92646357243417), np.float64(65.92204192397604), np.float64(65.91790021512689), np.float64(65.9139791647341), np.float64(65.91024236882805), np.float64(65.90664762553075), np.float64(65.9031462502141), np.float64(65.89968672066996), np.float64(

In [403]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-10:]]
    print(f"Tema {topic_idx+1}: {' '.join(top_words)}")

Tema 1: college beginning home baby money farm wood town family house
Tema 2: death horror killer apartment friend event truth night murder past
Tema 3: king death child strigoi war vampire family friend heart love
Tema 4: identity popularity wizard height witch land power case band adventure


In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

def prikazi_wordclouds(H, feature_names, n_top_words=10):
    n_topics = H.shape[0]
    
    
    plt.figure(figsize=(20, 10))
    
    for topic_idx, topic in enumerate(H):
        # slovar {beseda: utež} za WordCloud
        # top n besed za vsako temo
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        topic_words = {feature_names[i]: topic[i] for i in top_features_ind}
        
        
        wc = WordCloud(width=400, height=400, background_color='white', 
                       colormap='viridis', max_words=n_top_words)
        wc.generate_from_frequencies(topic_words)
    
        plt.subplot(2, 2, topic_idx + 1)
        plt.imshow(wc, interpolation='bilinear')
        plt.title(f"Tema {topic_idx + 1}", fontsize=20)
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

feature_names = vectorizer.get_feature_names_out()
prikazi_wordclouds(H, feature_names)

In [404]:
import pandas as pd

# W nam pove, kateri temi pripada knjiga
# dobimo indeks najvišje vrednosti v vsaki vrstici
dominant_topics = np.argmax(W, axis=1) + 1 


import os
filenames = [os.path.basename(f) for f in filepaths]

df_results = pd.DataFrame({
    'Knjiga': filenames,
    'Tema': dominant_topics
})

# prvih 10 rezultatov
print(df_results.head(50))

#specificne teme
print(df_results[df_results['Tema'] == 4])

          Knjiga  Tema
0     opis_1.txt     3
1    opis_10.txt     3
2   opis_100.txt     4
3   opis_101.txt     3
4   opis_102.txt     2
5   opis_103.txt     2
6   opis_104.txt     2
7   opis_105.txt     4
8   opis_106.txt     2
9   opis_107.txt     2
10  opis_108.txt     2
11  opis_109.txt     2
12   opis_11.txt     3
13  opis_110.txt     2
14  opis_111.txt     3
15  opis_112.txt     2
16  opis_113.txt     3
17  opis_114.txt     3
18  opis_115.txt     2
19  opis_116.txt     3
20  opis_117.txt     2
21  opis_118.txt     2
22  opis_119.txt     2
23   opis_12.txt     3
24  opis_120.txt     2
25  opis_121.txt     2
26  opis_122.txt     2
27  opis_123.txt     2
28  opis_124.txt     2
29  opis_125.txt     2
30  opis_126.txt     4
31  opis_127.txt     2
32  opis_128.txt     2
33  opis_129.txt     2
34   opis_13.txt     3
35  opis_130.txt     2
36  opis_131.txt     3
37  opis_132.txt     3
38  opis_133.txt     2
39  opis_134.txt     2
40  opis_135.txt     2
41  opis_136.txt     3
42  opis_13